# Team 10 - Member 8: self-contained late-fusion model (CASMI 2026)

Trains **everything inside this notebook**; it needs no other member's outputs.

1. **Library search**: cosine between a query's pooled spectrum and each structure's summed training spectra.
2. **Fingerprint net**: MLP, binned spectrum -> 2048-bit Morgan fingerprint (BCE). Candidates are ranked by fingerprint log-likelihood.
3. **Late fusion**: per-query normalisation (RRF), weights tuned on two validation sets: *absent* (held-out structures) and *covered* (leave-query-out).

Candidates are the structures seen in training, filtered by precursor mass (COCONUT can be added later).
Run with a **GPU** and the competition data attached. Internet is only needed if the repo is not attached as a dataset.

In [1]:
# 1) Find the repo code: attach it as a Kaggle Dataset (needs src/casmi inside), else clone it
# (the clone needs internet, a public repo and the code merged to the chosen branch).
import glob
import os
import subprocess
import sys

hits = glob.glob('/kaggle/input/**/src/casmi', recursive=True)
if hits:
    REPO = os.path.dirname(os.path.dirname(hits[0]))
else:
    REPO = '/kaggle/working/repo'
    if not os.path.isdir(REPO):
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main',
                        'https://github.com/ChaitanyaUppalapati/Team-10-Enveda-CASMI-2026.git', REPO], check=True)
for p in (os.path.join(REPO, 'src'), os.path.join(REPO, 'models', 'ensembles')):
    sys.path.insert(0, p)
print('repo:', REPO)
try:
    import rdkit
    print('rdkit', rdkit.__version__)
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rdkit'], check=True)

repo: /kaggle/input/datasets/aswin215/casmi-m8-code
rdkit 2026.09.2


In [2]:
# 2) Locate the competition files and set the config. Lower the caps for a quick dry run.
import glob

import torch
from ensemble_common.standalone import pipeline as P


def find(name):
    return next(iter(glob.glob(f'/kaggle/input/**/{name}', recursive=True)), None)


cfg = P.Config(
    train_path=find('train.parquet'), test_path=find('test.parquet'),
    # dry run: max_batches=3, max_absent_queries=200, max_covered_queries=200, max_fp_train=20000
)
print(cfg.train_path, cfg.test_path, 'cuda' if torch.cuda.is_available() else 'CPU ONLY - enable a GPU')

/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet /kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/test.parquet cuda


In [3]:
# 3) One pass over training: structures, fingerprints, library sums, validation queries, fingerprint-net data.
st = P.build_state(cfg)
print('fp train set:', st.fp_X.shape, ' library spectra:', int(st.library.counts.sum()))

275810 structures (27481 held out)
batch 1: library spectra=18045 fp_train=6882
batch 2: library spectra=36160 fp_train=12377
batch 3: library spectra=54371 fp_train=18012
batch 4: library spectra=72517 fp_train=23512
batch 5: library spectra=90528 fp_train=29015
batch 6: library spectra=108629 fp_train=34558
batch 7: library spectra=126877 fp_train=40129
batch 8: library spectra=144866 fp_train=45555
batch 9: library spectra=162676 fp_train=51123
batch 10: library spectra=180854 fp_train=56680
batch 11: library spectra=198813 fp_train=62200
batch 12: library spectra=216969 fp_train=67709
batch 13: library spectra=234804 fp_train=73086
batch 14: library spectra=252950 fp_train=78568
batch 15: library spectra=271082 fp_train=84110
batch 16: library spectra=288933 fp_train=89583
batch 17: library spectra=306954 fp_train=95086
batch 18: library spectra=325071 fp_train=100670
batch 19: library spectra=343029 fp_train=106153
batch 20: library spectra=361247 fp_train=111644
batch 21: library

In [4]:
# 4) Train the fingerprint network (GPU).
from ensemble_common.standalone.fpnet import TorchFingerprintModel

fp_model = TorchFingerprintModel(epochs=12, seed=cfg.seed).fit(st.fp_X, st.fp_Y)

/kaggle/input/datasets/aswin215/casmi-m8-code/models/ensembles/ensemble_common/standalone/fpnet.py:50: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  total += float(loss) * len(idx)


epoch 1/12 bce=0.0922
epoch 2/12 bce=0.0768
epoch 3/12 bce=0.0762
epoch 4/12 bce=0.0756
epoch 5/12 bce=0.0751
epoch 6/12 bce=0.0745
epoch 7/12 bce=0.0740
epoch 8/12 bce=0.0735
epoch 9/12 bce=0.0730
epoch 10/12 bce=0.0726
epoch 11/12 bce=0.0721
epoch 12/12 bce=0.0717


In [5]:
# 5) Validate: per-source MRR@25, candidate coverage, tuned fusion weights, fused MRR (absent / covered).
import json

report = P.tune_and_report(st, cfg, fp_model)
fused_val = report.pop('fused_scores')
print(json.dumps(report, indent=2))
os.makedirs('/kaggle/working/outputs', exist_ok=True)
json.dump(report, open('/kaggle/working/outputs/fusion_report.json', 'w'), indent=2)

absent: n=1500 coverage=0.615 {'library': 0.03043813032258718, 'fingerprint': 0.16779988315528765}
covered: n=1230 coverage=0.489 {'library': 0.33392861524318995, 'fingerprint': 0.12769279299650635}
{
  "per_source": {
    "absent": {
      "library": 0.03043813032258718,
      "fingerprint": 0.16779988315528765
    },
    "covered": {
      "library": 0.33392861524318995,
      "fingerprint": 0.12769279299650635
    }
  },
  "coverage": {
    "absent": 0.6153333333333333,
    "covered": 0.4894308943089431
  },
  "weights": {
    "library": 0.9819585505218156,
    "fingerprint": 0.018041449478184313
  },
  "method": "rrf",
  "fused_mrr_both": 0.16719905573788485,
  "fused": {
    "absent": 0.03043813032258718,
    "covered": 0.333980672098004
  }
}


In [6]:
# 6) Test predictions: top-25 candidates per molecule, fused with the tuned weights.
import pandas as pd

test_scores = P.predict_test(st, cfg, fp_model, report['weights'])
final = P.attach_smiles(test_scores, st)
final.to_csv('/kaggle/working/outputs/test_top25_with_smiles.csv', index=False)
test_scores.to_parquet('/kaggle/working/outputs/test_scores.parquet', index=False)
print(final.groupby('query_id').size().describe())
final.head()

test queries: 400
count    400.000000
mean      24.115000
std        3.570525
min        2.000000
25%       25.000000
50%       25.000000
75%       25.000000
max       25.000000
dtype: float64


,query_id,inchikey14,score,rank,smiles
0,m_005e53,QCTQNXJUUXYSRK,0.016371,1,CN(Cc1ncccc1C(=O)O)CC1(c2ccc(Br)cc2)CC1
1,m_005e53,XZGCKUAAHQVVHY,0.016134,2,O=C(O)c1cccnc1CN1CCC(c2ccccc2Br)CC1
2,m_005e53,SCIHCHFEKPRVSD,0.015864,3,CC1CC(c2ccc(Br)cc2)CN1Cc1ncccc1C(=O)O
3,m_005e53,ZSDDGNZHLINBRL,0.015584,4,Cc1nc(C2CC2)oc1C(=O)N1CCCc2c(Br)cccc2C1
4,m_005e53,BWNLJQYJJFXPQO,0.015361,5,Cc1ccc(CN2CC(C(=O)O)C(c3ccc(Br)cc3)C2)nc1


In [7]:
# 7) submission.csv. The sample file has one row per molecule; CHECK how it stores several SMILES before submitting.
sample_path = find('sample_submission.csv')
sample = pd.read_csv(sample_path) if sample_path else None
print(sample.head() if sample is not None else 'no sample_submission.csv found')
SMILES_SEP = ';'   # <-- set to match the sample (look at a row above); default is a GUESS
rows = (final.sort_values(['query_id', 'rank']).groupby('query_id')['smiles']
        .apply(lambda s: SMILES_SEP.join(s)).reset_index().rename(columns={'query_id': 'molecule_id'}))
if sample is not None:
    missing = set(sample.molecule_id) - set(rows.molecule_id)
    print('molecules without any candidate:', len(missing))
    fill = pd.DataFrame({'molecule_id': sorted(missing), 'smiles': ''})
    rows = pd.concat([rows, fill], ignore_index=True)
rows.to_csv('/kaggle/working/submission.csv', index=False)
print(rows.shape)

  molecule_id                                             smiles
0    m_005e53  CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CC...
1    m_006153  CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CC...
2    m_00b5aa  CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CC...
3    m_01e922  CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CC...
4    m_0259d4  CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CC...
molecules without any candidate: 0
(400, 2)
